# Positional Embedding

### vector embedding size for:
* #### GPT-2 small(117M parameters)= 768
* #### GPT-2 medium(345M parameters)= 1024
* #### GPT-2 large(762M parameters)= 1280
* #### GPT-2 XL(1.5B parameters)= 1600

## vocabulary size of GPT-2 pretraining is 50,257 created using Byte Pair Encoding(BPE)

In [15]:
import torch

import tiktoken

In [16]:
with open("verdict.txt",'r',encoding="utf-8") as f:
    raw=f.read()

In [17]:
vocab_size=50257
output_dim=256

In [18]:
token_embeding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [19]:
from torch.utils.data import Dataset,DataLoader

class tokenizerv1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:max_length+1+i]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, ind):
        return self.input_ids[ind],self.target_ids[ind]

In [20]:
def crearte_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=tokenizerv1(txt,tokenizer,max_length,stride)

    dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
    return dataloader

In [24]:
context=4
dataloader=crearte_dataloader_v1(raw,batch_size=8,max_length=context,stride=context-1,shuffle=False)

data_iter=iter(dataloader)
input,targets=next(data_iter)
print("\ninputs:\n",input)
print("\ntarget:\n",targets)



inputs:
 tensor([[10970, 33310,    35, 18379],
        [18379,   198,   198, 15749],
        [15749, 40417,   198,   198],
        [  198,    40,   550,  1464],
        [ 1464,  1807,  3619,   402],
        [  402,   271, 10899,  2138],
        [ 2138,   257,  7026, 15632],
        [15632,   438,  2016,   257]])

target:
 tensor([[33310,    35, 18379,   198],
        [  198,   198, 15749, 40417],
        [40417,   198,   198,    40],
        [   40,   550,  1464,  1807],
        [ 1807,  3619,   402,   271],
        [  271, 10899,  2138,   257],
        [  257,  7026, 15632,   438],
        [  438,  2016,   257,   198]])


##### current dimension is 8X4 after embedding to the token ids to embeding layer of 256 X 50257 we gat a 3D matrix of 8X4X265. Each token id will have 256 dimension vector atached to it.
##### 256 dimension vector ka matlab 256 dimension ka matrix nahi hota its just a vector or array of size 256.

In [27]:
token_embeding=token_embeding_layer(input)
print(token_embeding.shape)

torch.Size([8, 4, 256])


In [28]:
pos_embedding_layer=torch.nn.Embedding(context,output_dim)

In [29]:
 pos_embedding=pos_embedding_layer(torch.arange(context))
 print(pos_embedding.shape)

torch.Size([4, 256])


##### token_embeding(8X4X256) + pos_embedding(4X256)
##### input_embedding will be 8X4X256
##### 8 layers of pos_embedding is created to make it 8X4X256 from 4X256 so that it can be added to token_embedding